In [1]:
from os.path import join, dirname
import os
import sys
import random
from copy import deepcopy
from hydra import initialize, compose
from omegaconf import DictConfig
import numpy as np

import torch
import torch.nn.functional as F

sys.path.append(join(dirname(dirname(os.getcwd()))))
sys.path.append(os.getcwd())

from omniisaacgymenvs.utils.hydra_cfg.hydra_utils import *
from omniisaacgymenvs.utils.hydra_cfg.reformat import omegaconf_to_dict, print_dict
from omniisaacgymenvs.utils.task_util import initialize_task
from omniisaacgymenvs.envs.vec_env_rlgames import VecEnvRLGames

from memoryPerceiver.memory_perceiver.main.configuration import cf
import memoryPerceiver.memory_perceiver.isaac.trainer as tr
from memoryPerceiver.memory_perceiver.isaac.plot import Plotter
from memoryPerceiver.memory_perceiver.isaac.agent import ActorCritic, ActorCriticSimple, ActorCriticPerceiver

%load_ext autoreload
%autoreload 2
%env ISAAC_JUPYTER_KERNEL=1

env: ISAAC_JUPYTER_KERNEL=1


/isaac-sim/kit/python/lib/python3.7/site-packages/IPython/extensions/autoreload.py:121: DeprecationWarning: the imp module is deprecated in favour of importlib; see the module's documentation for alternative uses
  from imp import reload


In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device_cpu = torch.device("cpu")
random.seed(0)
np.random.seed(0)
torch.use_deterministic_algorithms(False)
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

env: CUBLAS_WORKSPACE_CONFIG=:4096:8


In [3]:
"""model = ActorCriticSimple(cf, device)
model_student = ActorCriticSimple(cf, device)
#actions, logs, vals, dist = model.act(torch.randn(8,4, device=device))
#print(actions.size(), logs.size(), vals.size())
"""

'model = ActorCriticSimple(cf, device)\nmodel_student = ActorCriticSimple(cf, device)\n#actions, logs, vals, dist = model.act(torch.randn(8,4, device=device))\n#print(actions.size(), logs.size(), vals.size())\n'

In [4]:
model = ActorCriticPerceiver(cf, device)
model_student = ActorCriticPerceiver(cf, device)
numEnvs=1
states = {'observations' : torch.randn(numEnvs,cf.frames,4, device=device),
         'video' : torch.zeros(numEnvs, 3, cf.frames, 256, 456, device=device),
         'audio':torch.zeros(numEnvs, cf.frames*1920, device=device)
        }
model.act(states)

(tensor([[0.3308, 0.2824]], device='cuda:0'),
 tensor([-0.9724], device='cuda:0'),
 tensor([-0.0302], device='cuda:0'),
 MultivariateNormal(loc: torch.Size([1, 2]), covariance_matrix: torch.Size([1, 2, 2])))

In [5]:
"""
do not care about memory being the same, backpropagate such that score are good
"""
#@hydra.main(config_name="config", config_path="../cfg")
#@hydra.main(config_name="config", config_path="../../OmniIsaacGymEnvs/omniisaacgymenvs/cfg")
def parse_hydra_configs(cfg: DictConfig):
    cfg_dict = omegaconf_to_dict(cfg)

    render = True
    env = VecEnvRLGames(headless=cfg.headless, sim_device=cfg.device_id, enable_viewport=cfg.enable_viewport)
    
    
    # !!! Load isaac modules after the variables
    from omni.isaac.core.utils.torch.maths import set_seed
    import omni.usd
    # from omni.isaac.core.utils.stage import open_stage, save_stage
    
    plotter = Plotter(titles=['loss_actor', 'loss_critic', 'rewards'])
    history_keys = ['actions', 'values', 'logprobs', 'returns', 'rewards', 'advantages', 'death_index', 'generator']
    history = tr.init_history(history_keys, sim_duration, numEnvs, cf.body_outputs['actions'], device)
    previous_history = deepcopy(history)
    buffer = {'video':[], 'audio':[], 'obs':[], 'rewards':[], 'actions_star':[]}
    memories = {i:None for i in range(numEnvs)}

    cfg.seed = set_seed(cfg.seed, torch_deterministic=cfg.torch_deterministic)
    cfg_dict['seed'] = cfg.seed  # sets seed. if seed is -1 will pick a random one
    task = initialize_task(cfg_dict, env)
    all_losses_actor, all_losses_critic, all_returns = [], [], []
    optimizer = torch.optim.Adam(model_student.parameters(), lr=learning_rate)
    sim_count, sim_step = 0, 0
    first_run=True
    is_inference=False  # Will switch to True immediately
    
    while env._simulation_app.is_running():
        if env._world.is_playing():
            # if env._world.current_time_step_index == 0 or \
            # env._world.current_time_step_index % sim_duration == 0:
            if env.sim_frame_count % sim_duration == 0:

                ### Plots

                if sim_count % 6 == 1 and len(all_losses_actor)>0:
                    plotter.plot_all([(all_losses_actor, 160), (all_losses_critic, 160), all_returns])

                ### Initialization

                env._world.reset(soft=False)
                is_inference = not is_inference
                env._task.post_reset()
                print('$$$$$$$$$$$ CUMUL REWARD $$$$$$$$$$$')
                if len(history['rewards'])>0:
                    print(history['rewards'].sum())
                if is_inference:
                    print('PLAY')
                    if first_run and os.path.exists(path_model):
                        model.load_state_dict(torch.load(path_model))
                        model_student.load_state_dict(torch.load(path_model))
                    else:
                        if sim_count % student_update_step*2 == 0:
                            model.load_state_dict(model_student.state_dict())
                    # model.reset_memories()
                    torch.save(model_student.state_dict(), path_model)
                    model.eval()
                else:
                    print('REPLAY')
                    first_run=False
                    # model.reset_memories()
                    # model_student.reset_memories()
                    model_student.train() 
                    optimizer.zero_grad()
                    previous_history = deepcopy(history)
                    previous_history['returns'] = tr.calculate_returns(history['rewards'], history['death_index'], gamma=c_gamma)
                    previous_history['advantages'] = tr.calculate_gae(history['rewards'], history['values'], 
                                                        history['death_index'], gamma=c_gamma, gae_lambda=c_lambda)
                    all_returns.append(previous_history['returns'][1, :].mean().item())
                    
                torch.manual_seed(sim_count//2)
                torch.cuda.manual_seed(sim_count//2)
                memories = {i:None for i in range(numEnvs)}
                birth_time = torch.zeros(numEnvs)
                for key in buffer.keys():
                    buffer[key]=[]
                history = tr.init_history(history_keys, sim_duration, numEnvs, cf.dim_actions_ppo, device)
                backward_count, kl_div = 0, 0
                sim_count+=1
                sim_step = 0

            ### Get agent observations 

            all_obs = env._task.get_observations()['cartpole_view']
            obs = all_obs['obs_buf'].clone()    # (numEnvs, 4)
            video = all_obs['video_buf'].clone()  # (numEnvs, 3, 256, 456)
            env._task.calculate_metrics()       
            reward=env._task.rew_buf.clone()    # (numEnvs, 1)
            tr.update_buffers(buffer, {'obs': obs, 'video': video, 'rewards': reward})
            history['rewards'][sim_step] = reward

            ### Reset goal and memory at death

            for agent_i in range(numEnvs):
                if env._task.reset_buf[agent_i].item() == 1:
                    memories[agent_i] = None
                    birth_time[agent_i] = sim_step

                    # Update death dictionnary during inference

                    if is_inference:
                        history['death_index'][sim_step, agent_i]=1


            ### Process input

            states={}
            states['video'] = torch.zeros(numEnvs, 3, cf.frames, 256, 456, device=device)
            # inputs['video'] = rearrange(buffer_padding(buffer['video'], [3, 256, 456]), 'b t c h w -> b c t h w')
            states['audio'] = torch.zeros(numEnvs, cf.frames*1920, device=device)

            ### Agent
            
            if len(buffer['actions_star'])==0 :
                states['observations'] = tr.buffer_padding(torch.stack(buffer['obs']), 
                                                           [cf.body_parts['observations']], numEnvs=numEnvs)
                # mask = create_mask(birth_time, sim_step)
                # inputs['observations'][mask] *= 0

                ## Inference, save history and memory update

                if is_inference:
                    actions, logprobs, values, old_dist = model.act(states)   
                    history['actions'][sim_step] = actions  
                    history['logprobs'][sim_step] = logprobs 
                    history['values'][sim_step] = values     
                    history['generator'][sim_step] = torch.cuda.get_rng_state().clone()
                else:
                    actions, logprobs, values = [previous_history[key][sim_step] for key in ['actions', 'logprobs', 'values']]
                    torch.cuda.set_rng_state(previous_history['generator'][sim_step])
                                    

                ## Backward pass

                if not is_inference and sim_step>0:

                    returns = previous_history['returns'][sim_step]
                    advantages = previous_history['advantages'][sim_step]
                    new_logprobs, new_values, entropies, new_dist = model_student.evaluate(states, actions)

                    # Loss
                    ratios = torch.exp(new_logprobs - logprobs)
                    surr1 = ratios * advantages
                    surr2 = torch.clamp(ratios, 1-eps_clip, 1+eps_clip) * advantages
                    loss_actor = -(torch.min(surr1, surr2)).mean()
                    loss_entropy = -c_entropy * entropies.mean()
                    loss_critic = c_critic * F.mse_loss(new_values, returns)
                    loss =  (loss_actor + loss_critic + loss_entropy) /accumulation_steps
                    loss.backward()
                    backward_count += 1
                    kl_div += torch.distributions.kl_divergence(new_dist, old_dist).mean()/accumulation_steps


                    # Step optimizer

                    if backward_count % accumulation_steps == 0 and backward_count != 0:
                        torch.nn.utils.clip_grad_norm_(model_student.parameters(), c_grad_norm)
                        optimizer.step()
                        optimizer.zero_grad()

                        # Adaptive schedule
                        for param_group in optimizer.param_groups:
                            if adaptive_schedule:
                                param_group['lr'] = tr.update_lr(param_group['lr'], kl_div)

                        # Reinitialize
                        backward_count = 0
                        kl_div = 0


                    all_losses_actor.append(loss_actor.item())
                    all_losses_critic.append(loss_critic.item())

                # Update buffers

                actions_star = model.instanciate_actions(actions)
                buffer['actions_star'] += list(torch.unbind( actions_star ))


            ## Current action (numEnvs, 1)

            if(len(buffer['actions_star'])>0):
                action = buffer['actions_star'].pop(0)
                action = torch.clamp(action, min=-1, max=1)
            else:
                action = torch.dzeros(numEnvs, 1)

            # if( inference_done ):
            #     print('$$$$$$  INFERENCE  $$$$$$')
            #     print('obs ', inputs['observations'][agent_i,-1])
            #     if replay_training:
            #         print('advantages: ', advantages)

            # if env.sim_frame_count % sim_duration == 1:
            #     print("$$$$$$$$$$$$ 00000 $$$$$$$$$$$")
            #     if replay_training:
            #         print('returns: ', returns_tmp)

            # Step environment

            env._task.pre_physics_step(action.to(task.rl_device))
            env._world.step(render=render)
            env.sim_frame_count += 1
            env._task.post_physics_step()

            sim_step+=1

        else:
            env._world.step(render=render)

    env._simulation_app.close()


if __name__ == '__main__':
    sim_duration = 97
    numEnvs = 1
    learning_rate = 0.0001
    accumulation_steps = 4
    c_entropy = 0
    c_critic=4
    student_update_step = 1
    eps=1e-9
    c_gamma=0.99
    c_lambda=0.95
    c_grad_norm=1.0
    eps_clip=0.2
    adaptive_schedule=False
    path_model = 'weights/cartpole.pt'
    critic_path = 'weights/critic.pt'
    with initialize(config_path="memoryPerceiver/OmniIsaacGymEnvs/omniisaacgymenvs/cfg"):
        cfg = compose(config_name="config", overrides=["task=Cartpole", "headless=True","enable_viewport=True",
                        "+numEnvs="+str(numEnvs)])
    parse_hydra_configs(cfg)

/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config': Defaults list is missing `_self_`. See https://hydra.cc/docs/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:412: UserWarning: In config: Invalid overriding of hydra/job_logging:
Default list overrides requires 'override' keyword.
See https://hydra.cc/docs/next/upgrades/1.0_to_1.1/defaults_list_override for more information.

  deprecation_warning(msg)


Starting kit application with the following args:  ['/isaac-sim/exts/omni.isaac.kit/omni/isaac/kit/simulation_app.py', '/isaac-sim/apps/omni.isaac.sim.python.gym.headless.render.kit', '--/app/tokens/exe-path=/isaac-sim/kit', '--/persistent/app/viewport/displayOptions=3094', '--/rtx/materialDb/syncLoads=True', '--/rtx/hydra/materialSyncLoads=True--/omni.kit.plugin/syncUsdLoads=True', '--/app/renderer/resolution/width=1280', '--/app/renderer/resolution/height=720', '--/app/window/width=1440', '--/app/window/height=900', '--/renderer/multiGpu/enabled=True', '--/app/fastShutdown=False', '--ext-folder', '/isaac-sim/exts', '--ext-folder', '/isaac-sim/apps', '--/physics/cudaDevice=0', '--portable', '--no-window', '--allow-root']
Passing the following args to the base kit application:  ['-f', '/root/.local/share/jupyter/runtime/kernel-449d9cbb-30bd-4676-a552-c0a4cc84eb89.json']
[Warning] [omni.isaac.kit.simulation_app] Modules: ['omni.isaac.kit.app_framework'] were loaded before SimulationApp 

/isaac-sim/kit/kernel/py/omni/kit/app/_impl/__init__.py:104: DeprecationWarning: Using or importing the ABCs from 'collections' instead of from 'collections.abc' is deprecated since Python 3.3,and in 3.9 it will stop working
  if not isinstance(setting_py, str) and isinstance(setting_py, collections.Sequence):


[4.891s] [ext: omni.kit.viewport.registry-104.0.2] startup
[4.895s] [ext: omni.kit.menu.create-1.0.8] startup
[4.899s] [ext: omni.kit.material.library-1.3.21] startup
[4.912s] [ext: omni.kit.viewport.legacy_gizmos-1.0.10] startup
[4.934s] [ext: omni.kit.hydra_texture-1.0.11] startup
[4.948s] [ext: omni.kit.context_menu-1.5.12] startup
[4.955s] [ext: omni.ui.scene-1.5.18] startup
[4.968s] [ext: omni.kit.window.drop_support-1.0.1] startup
[4.969s] [ext: omni.kit.widget.viewport-104.1.16] startup
[4.976s] [ext: omni.kit.window.preferences-1.3.8] startup
[5.236s] [ext: omni.usdphysics-104.2.4-5.1] startup
[5.244s] [ext: omni.kit.viewport.window-104.1.23] startup
2024-02-21 16:44:12 [5,223ms] [Warning] [omni.kit.ui] Failed to set value for menu with path: Window/Viewport/Viewport 1. It doesn't exist.
[5.313s] [ext: omni.kit.manipulator.transform-104.6.12] startup
[5.318s] [ext: omni.physx-104.2.4-5.1] startup
[5.390s] [ext: omni.kit.viewport.utility-1.0.14] startup
[5.391s] [ext: omni.kit.w

/isaac-sim/kit/exts/omni.graph/omni/graph/core/_impl/autonode/type_definitions.py:12: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  Float = NewType("Float", np.float)


[5.790s] [ext: omni.graph.nodes-1.48.3] startup
[5.817s] [ext: omni.kit.viewport.menubar.settings-104.3.5] startup
[5.832s] [ext: omni.kit.viewport.menubar.render-104.0.20] startup
[5.838s] [ext: omni.isaac.core-1.46.3] startup
Warp 0.6.3 initialized:
   CUDA Toolkit: 11.5, Driver: 12.2
   Devices:
     "cpu"    | x86_64
     "cuda:0" | NVIDIA GeForce RTX 3060 (sm_86)
   Kernel cache: /root/.cache/warp/0.6.3
[6.268s] [ext: omni.warp-0.6.3] startup
[7.031s] [ext: omni.kit.window.title-1.1.2] startup
2024-02-21 16:44:14 [7,006ms] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2024-02-21 16:44:14 [7,006ms] [Warning] [carb] Failed to startup plugin carb.windowing-glfw.plugin (interfaces: [carb::windowing::IGLContext v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
2024-02-21 16:44:14 [7,007ms] [Warning] [carb] [Plugin: libomni.structuredlog.plugin.so] Module /isaac-sim/kit/libomni.structuredlog.plugin.so remained loaded after unload request
2

/isaac-sim/kit/python/lib/python3.7/site-packages/gym/spaces/box.py:84: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


2024-02-21 16:44:21 [14,020ms] [Warning] [omni.client.plugin]  Tick: authentication: Could not resolve host name "isaac-dev.ov.nvidia.com"
2024-02-21 16:44:21 [14,025ms] [Warning] [omni.client.plugin]  Tick: authentication: Could not resolve host name "isaac-dev.ov.nvidia.com"
2024-02-21 16:44:21 [14,044ms] [Warning] [omni.client.plugin]  Tick: authentication: Could not resolve host name "isaac-dev.ov.nvidia.com"
2024-02-21 16:44:21 [14,045ms] [Warning] [omni.isaac.core.utils.nucleus] /persistent/app/omniverse/mountedDrives setting not found
2024-02-21 16:44:21 [14,362ms] [Warning] [omni.isaac.core.utils.nucleus] /persistent/app/omniverse/mountedDrives setting not found
2024-02-21 16:44:22 [14,655ms] [Warning] [omni.replicator.core.scripts.writers] Writer already exists. Overwriting writer PytorchWriter.
~*~*~*
~*~*~* Direct GPU Helper:
~*~*~*   1 articulation, maxLinks=3
~*~*~*
2024-02-21 16:44:24 [17,100ms] [Warning] [omni.replicator.core.scripts.writers] Unable to write metadata, no

In [ ]:
!nvidia-smi

Sat Feb 10 12:32:36 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.154.05             Driver Version: 535.154.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA RTX A2000 12GB          On  | 00000000:06:00.0 Off |                  Off |
| 60%   42C    P8              12W /  70W |      1MiB / 12282MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--